In [1]:
from datasets import cohorts_map, CATEGORICAL_CLINICAL_VARIABLES

loader = cohorts_map["headneckpetct"]
imaging, clinical, rfs = loader(None).get_features_labels("./")

In [7]:
for T in (2, 5):
    print(f"{T} years")
    labels = []
    for _, rfs_p in rfs.items():
        if rfs_p["T"] is None:
            continue
        elif rfs_p["T"] < T*365:
            if rfs_p["delta"] == 1:
                labels.append(1)
            else:
                continue
        else:
            labels.append(0)

    print("total patients ", len(rfs))
    print("patients after label compute ", len(labels))
    for i in set(labels):
        print(f"{i} ({int(100*labels.count(i)/len(labels))}%)")

2 years
total patients  298
patients after label compute  277
0 (79%)
1 (20%)
5 years
total patients  298
patients after label compute  121
0 (40%)
1 (59%)


In [2]:
for i in list(clinical.values())[0].keys():
    print(i)

sex
age
localisation
metastasis
stage
hpv
treatment
surgery
dose
volume


In [4]:
import numpy as np

def print_numerical(key, clinical):
    data = [i[key] for i in clinical.values() if not(i[key] is None)]
    m, p10, p90 = int(np.mean(data)), int(np.percentile(data, 10)), int(np.percentile(data, 90))
    print(f"\t {m} [{p10}-{p90}]")

def print_categorical(key, clinical):
    data = [i[key] for i in clinical.values()]
    for i in set(data):
        print(f"\t {i}: {data.count(i)} ({100*(data.count(i)/len(data)):.1f}%)")

def demographics(clinical, rfs):
    print("age")
    print_numerical("age", clinical)

    print("GTV dose")
    print_numerical("dose", clinical)

    print("GTV volume (voxels^3)")
    print_numerical("volume", clinical)

    # "Female": 0, "Male": 1
    print("sex")
    print_categorical("sex", clinical)

    print("TNM stage")
    print_categorical("stage", clinical)

    # "RT alone": 0, "ChemoRT": 1
    print("treatment")
    print_categorical("treatment", clinical)

demographics(clinical, rfs)

age
	 63 [51-77]
GTV dose
	 66 [51-73]
GTV volume (voxels^3)
	 39195 [9255-76615]
sex
	 0: 71 (23.8%)
	 1: 227 (76.2%)
TNM stage
	 1: 4 (1.3%)
	 2: 27 (9.1%)
	 3: 61 (20.5%)
	 4: 204 (68.5%)
	 None: 2 (0.7%)
treatment
	 0: 48 (16.1%)
	 1: 250 (83.9%)


In [35]:
from scipy import stats
from scipy.stats import fisher_exact
import pandas

def filter_nan(values):
    return list(filter(lambda i: not(i is None or pandas.isna(i)), values))

def split_variable_groups(T, var, rfs, clinical):
    pos, neg = [], []
    for p, rfs_p in rfs.items():
        try:
            if rfs_p["T"] is None:
                continue
            elif rfs_p["T"] < T*365:
                if rfs_p["delta"] == 1:
                    pos.append(clinical[p][var])
                else:
                    continue
            else:
                neg.append(clinical[p][var])
        except KeyError:
            continue
    pos = filter_nan(pos)
    neg = filter_nan(neg)
    return pos, neg

def numerical_pvalue(T, var, rfs, clinical):
    pos, neg = split_variable_groups(T, var, rfs, clinical)
    _, p_val = stats.ttest_ind(pos, neg)
    return p_val

def categorical_pvalue(T, var, rfs, clinical):
    pos, neg = split_variable_groups(T, var, rfs, clinical)
    df_a = pd.DataFrame({'rfs': 'pos', 'outcome': pos})
    df_b = pd.DataFrame({'rfs': 'neg', 'outcome': neg})
    df = pd.concat([df_a, df_b])
    contingency_table = pd.crosstab(df['rfs'], df['outcome'])
    _, p_value = fisher_exact(contingency_table.values)
    return p_value

for T in (2,5):
    print(f"{T} years")
    for var in ("age", "dose", "volume"):
        print(var)
        print(numerical_pvalue(T, var, rfs, clinical))

    for var in ("sex", "stage", "treatment"):
        print(var)
        print(categorical_pvalue(T, var, rfs, clinical))

2 years
age
0.18094879981933426
dose
0.6489441845430237
volume
2.592730266256936e-05
sex
0.1601630428201984
stage
0.6417
treatment
0.8358520306151984
5 years
age
0.506652351346266
dose
0.9497397942374759
volume
0.028589055760758098
sex
0.2711817199055696
stage
0.4791
treatment
0.34431458847743146
